# Лабораторная работа № 7
## Реализация выбора лидера и механизма консенсуса в распределённой системе

### Цель работы
Изучить принципы согласования состояния между узлами распределённой системы и понять, как реализуются выбор лидера, обнаружение отказов, повторные выборы, голосование и принятие решения на основе большинства.

Лабораторная работа продолжает проект из предыдущих работ. Необходимо сохранить предметную область и вариант, назначенный в ЛР № 1. Полноценные реализации Paxos, Raft или Zab не требуются: реализуется упрощённая учебная модель.

### Основной сценарий
**Выбор лидера → обнаружение отказа → повторный выбор лидера → голосование → достижение консенсуса.**

## 1. Распределённая система и узлы

Распределённая система состоит из нескольких взаимодействующих узлов, каждый из которых является отдельным компонентом. Узлы могут обмениваться сообщениями и участвовать в выполнении общей задачи.

Если узлы независимо принимают решения о состоянии системы, возможны противоречия: разные участники могут считать актуальными разные значения. Поэтому необходим механизм согласования.

В данной работе используется не менее пяти узлов: `Node 1` — `Node 5`. Каждый узел должен иметь уникальный идентификатор, состояние, роль, сведения о других узлах, возможность отправлять и принимать сообщения и изменять своё состояние.

## 2. Консенсус (Consensus)

**Консенсус** — процесс, с помощью которого несколько узлов распределённой системы приходят к согласованному решению. В контексте лабораторной работы узлы голосуют за предложенное значение, а решение считается принятым, если получено необходимое большинство положительных голосов.

Например, лидер предлагает `ConfigurationVersion = 15`. Если три из пяти узлов голосуют `YES`, то в рамках учебной модели консенсус достигнут. Если положительных голосов меньше необходимого большинства, решение принимать нельзя.

Важно различать выбор лидера и консенсус: выбор лидера определяет, кто координирует операцию; консенсус определяет, согласились ли узлы с конкретным предлагаемым значением.

## 3. Лидер и ведомые узлы

**Leader (лидер)** — узел, которому поручена координация определённых операций. Он может координировать действия, распространять информацию, предлагать изменения состояния, собирать голоса и объявлять результат консенсуса.

**Follower (ведомый узел)** — узел, который не является текущим лидером, но продолжает участвовать в работе системы. Он получает сообщения, отвечает на проверки доступности, участвует в выборах, рассматривает предложения и отдаёт голос.

Отказ лидера не должен приводить к аварийному завершению всей системы. При наличии достаточного количества доступных узлов система должна попытаться выбрать нового лидера.

## 4. Выбор лидера (Leader Election)

**Leader election** — процедура выбора узла, который будет выполнять роль лидера.

В данной лабораторной работе применяется простое детерминированное правило: лидером становится **доступный узел с максимальным ID**.

Пример:
- `Node 1` — `AVAILABLE`
- `Node 2` — `AVAILABLE`
- `Node 3` — `AVAILABLE`
- `Node 4` — `AVAILABLE`
- `Node 5` — `AVAILABLE`

Лидером становится `Node 5`. Если `Node 5` получает состояние `FAILED`, он исключается из кандидатов, а новым лидером становится `Node 4`, если тот доступен.

У каждого узла есть состояние и роль. В работе используются состояния `AVAILABLE` и `FAILED`, а также роли `LEADER` и `FOLLOWER`. В корректном состоянии системы одновременно должен быть не более чем один действующий лидер.

## 5. Обнаружение отказов и Heartbeat

Распределённая система должна определять, отвечает ли узел на запросы. Для учебной модели используется **heartbeat** — сообщение-проверка доступности.

Пример обмена:
```text
Node 1 → Node 5: HEARTBEAT?
Node 5 → Node 1: ALIVE
```

Если ответ не приходит в отведённое время, возникает тайм-аут (`heartbeat timeout`), после которого узел можно считать недоступным. В лабораторной работе отказ разрешено моделировать изменением состояния, остановкой обработки, искусственным исключением или прекращением ответов на heartbeat.

Тайм-аут сам по себе не всегда доказывает физический отказ: причиной может быть задержка сети или перегрузка. Поэтому в реальных системах обнаружение отказа сложнее, чем простая проверка отсутствия ответа.

## 6. Повторный выбор лидера

Если текущий лидер становится недоступным, система фиксирует отказ и запускает выборы заново. Недоступный узел не должен быть выбран лидером.

Пример последовательности:
```text
Node 5 is LEADER
Node 5 heartbeat timeout
Node 5 → FAILED
Election started
Node 5 excluded
Node 4 elected as LEADER
```

После выборов нужно обновить роли узлов: новый лидер получает роль `LEADER`, остальные доступные узлы — `FOLLOWER`, а недоступные остаются `FAILED`. Следует проверить, что одновременно не назначены два действующих лидера.

## 7. Большинство (Majority) и кворум

**Большинство** — число голосов, превышающее половину общего количества узлов. Для фиксированного количества узлов \(N\) минимальное строгое большинство вычисляется так:

\[
M = \left\lfloor \frac{N}{2} \right\rfloor + 1
\]

Для пяти узлов требуется три голоса: \(M = \lfloor 5/2 \rfloor + 1 = 3\). В контексте распределённых алгоритмов такое необходимое большинство часто называют **кворумом (quorum)**.

Для системы из пяти узлов:
- доступно 5 узлов — большинство достижимо;
- доступно 4 узла — большинство достижимо;
- доступно 3 узла — большинство достижимо;
- доступно 2 или 1 узел — получить большинство из пяти невозможно.

Таким образом, учебная система может продолжать принимать решения после части отказов, пока сохраняется необходимое большинство. Важно, чтобы правило подсчёта голосов было определено однозначно: в этой лабораторной работе большинство рассчитывается от общего числа узлов, а недоступные узлы не дают положительного голоса.

## 8. Предложение значения и голосование

После выбора лидера реализуется процедура согласования некоторого значения. Лидер формирует предложение — например, `ConfigurationVersion = 10` — и отправляет его доступным узлам.

Каждый доступный узел возвращает один из ответов:
- `YES` — узел поддерживает предложение;
- `NO` — узел отклоняет предложение;
- отсутствие ответа — узел не участвует в голосовании как ответивший участник.

Лидер подсчитывает положительные голоса и сравнивает их с порогом большинства. Например, при пяти узлах три ответа `YES` достаточны для достижения консенсуса. Если получено только два положительных голоса, консенсус не достигнут.

Недоступный узел не должен голосовать, а отсутствие большинства не должно приводить к принятию решения.

## 9. Журналирование

**Журнал работы (log)** фиксирует последовательность событий, чтобы можно было понять, как система пришла к результату и что произошло при отказе.

В журнале должны отображаться:
- время события;
- идентификатор узла, его состояние и роль;
- начало выборов и результат выбора лидера;
- обнаружение отказа и тайм-аут heartbeat;
- повторный выбор лидера;
- предложенное значение;
- голоса узлов;
- число положительных голосов и необходимое большинство;
- результат: `CONSENSUS ACHIEVED` или `CONSENSUS NOT ACHIEVED`.

Пример:
```text
[14:30:01.110] Election started
[14:30:01.120] Node 5 elected as LEADER
[14:30:05.100] Node 5 → HEARTBEAT TIMEOUT
[14:30:05.110] Node 5 → FAILED
[14:30:05.130] Node 4 elected as LEADER
[14:30:10.100] Leader Node 4 → PROPOSE Version=12
[14:30:10.110] Node 1 → YES
[14:30:10.120] Node 2 → YES
[14:30:10.130] Node 3 → YES
[14:30:10.150] YES = 3; Majority = 3
[14:30:10.152] CONSENSUS ACHIEVED
```

## 10. Обработка ошибок и отказоустойчивость

Приложение должно корректно обрабатывать недоступность одного или нескольких узлов, отказ текущего лидера, отсутствие ответа на heartbeat, невозможность связи, отсутствие большинства, попытку голосования недоступного узла, повторный запуск выборов, одновременное получение сообщений и некорректные сообщения.

**Отказоустойчивость** — способность системы продолжать работу при отказе отдельных компонентов в пределах предусмотренных ограничений. Для этой модели ключевые правила таковы:
1. отказ лидера запускает повторные выборы, если это возможно;
2. недоступный узел исключается из кандидатов и не голосует;
3. при отсутствии необходимого большинства система не объявляет консенсус;
4. ошибка отдельного узла не должна без необходимости завершать всё приложение.

## 11. Связь с Paxos, Raft и Zab

В лабораторной работе требуется связать реализованную модель с известными алгоритмами, но **не реализовывать их полностью**.

- **Raft** — алгоритм консенсуса, в котором явно выделяются роли узлов и предусмотрен выбор лидера. Поэтому учебная модель выбора лидера и его повторного выбора наиболее наглядно сопоставляется с Raft.
- **Paxos** — семейство алгоритмов консенсуса, использующих предложения и обмен сообщениями между участниками для согласования значения при возможных отказах. Модель голосования иллюстрирует общую задачу согласования, но не является реализацией Paxos.
- **Zab (ZooKeeper Atomic Broadcast)** — протокол, используемый Apache ZooKeeper для согласования состояния и упорядочивания изменений. Его назначение связано с поддержанием согласованного порядка обновлений в распределённой системе.

Простое правило «доступный узел с максимальным ID» и подсчёт голосов — учебная модель. Она не включает все состояния, сообщения, гарантии безопасности и восстановления, предусмотренные промышленными алгоритмами.

## 12. Эксперименты с отказами

Для системы из пяти узлов нужно выполнить четыре сценария:

| Сценарий | Доступно узлов | Требуемое большинство | Ожидаемый результат |
|---|---:|---:|---|
| Все узлы доступны | 5 из 5 | 3 | Консенсус возможен |
| Отказ одного узла | 4 из 5 | 3 | Консенсус возможен |
| Отказ двух узлов | 3 из 5 | 3 | Консенсус возможен |
| Отказ трёх узлов | 2 из 5 | 3 | Консенсус невозможен |

В каждом эксперименте следует зафиксировать состояние узлов, выбранного лидера, голоса и итоговый результат. Если доступно только два узла из пяти, приложение может провести попытку голосования, но не должно принимать предложение как согласованное большинством.

## 13. Unit-тестирование

**Unit-тестирование** проверяет небольшие компоненты программы изолированно, чтобы подтвердить корректность их поведения.

Минимальный набор проверок по заданию:
- создание узлов и уникальность ID;
- определение доступных узлов;
- выбор лидера и невозможность выбрать `FAILED`-узел;
- обнаружение отказа лидера и повторные выборы;
- вычисление большинства;
- положительное и отрицательное голосование;
- достижение консенсуса при достаточном числе голосов;
- невозможность консенсуса без большинства;
- обработка отказа и корректная работа после смены лидера.

Тесты должны проверять не только успешные сценарии, но и граничные случаи: нет доступных узлов, доступно меньше кворума, лидер отказал во время операции, пришло некорректное сообщение.

## 14. Роли компонентов приложения

**Узел** хранит ID, состояние и роль, знает другие узлы, отвечает на heartbeat, участвует в выборах и голосовании, журналирует события и обрабатывает ошибки.

**Лидер** координирует операцию, отправляет предложение, принимает и подсчитывает голоса, проверяет наличие большинства и объявляет результат.

**Follower** получает сообщения, отвечает на heartbeat, участвует в выборах, рассматривает предложение и возвращает голос.

**Клиент** подключается к серверу, инициирует операцию из предыдущих лабораторных работ, отображает результат и ошибки. Проект должен оставаться продолжением предыдущей работы и сохранять исходную предметную область.

## 15. Что нужно уметь объяснить на защите

1. Что такое узел распределённой системы и почему требуется согласование состояния.
2. Что такое консенсус, лидер и ведомый узел.
3. Как работает выбор лидера по максимальному доступному ID.
4. Для чего нужен heartbeat и как трактуется тайм-аут.
5. Что происходит после отказа лидера.
6. Как вычисляется большинство и почему для пяти узлов нужно три голоса.
7. Как предложение проходит голосование и при каких условиях принимается.
8. Почему наличие лидера не гарантирует консенсус.
9. Чем учебная модель связана с Paxos, Raft и Zab и почему она не является полной реализацией этих алгоритмов.
10. Какие сценарии отказов проверяются и что подтверждают Unit-тесты.

## Контрольные вопросы с краткими ответами

**1. Что такое консенсус?**  
Процесс достижения согласованного решения несколькими узлами распределённой системы.

**2. Для чего необходим консенсус?**  
Чтобы участники согласованно определяли принимаемое значение или состояние и не принимали противоречивые решения.

**3. Что такое Leader?**  
Узел, координирующий определённые операции, например рассылку предложения и сбор голосов.

**4. Что такое Follower?**  
Узел, который не является текущим лидером, но участвует в обмене сообщениями, выборах и голосовании.

**5. Что такое Leader election?**  
Процедура определения узла, который будет лидером.

**6. Что происходит при отказе лидера?**  
Отказ фиксируется, недоступный узел исключается, после чего запускаются повторные выборы.

**7. Для чего используется Heartbeat?**  
Для периодической проверки того, отвечает ли узел.

**8. Как определить, что узел недоступен?**  
В учебной модели — по отсутствию ответа в заданный промежуток времени или по явно установленному состоянию `FAILED`.

**9. Что такое Majority?**  
Строгое большинство участников; для пяти узлов это три голоса.

**10. Что такое голосование?**  
Ответ узла `YES` или `NO` на предложенное значение.

**11. Что произойдёт, если голоса разделились поровну?**  
Если положительных голосов недостаточно для требуемого большинства, консенсус не достигнут.

**12. Почему лидер сам по себе не гарантирует консенсус?**  
Потому что лидер может предложить значение, но доступные узлы могут его отклонить или их голосов может оказаться недостаточно.

**13. Чем выбор лидера отличается от консенсуса?**  
Выбор лидера определяет координатора; консенсус означает принятие конкретного предложения необходимым большинством.

**14. Что такое отказ узла?**  
Состояние, при котором узел перестаёт выполнять ожидаемые функции или отвечать на сообщения.

**15. Что такое Paxos?**  
Семейство алгоритмов, позволяющих распределённым участникам согласовать значение при возможных отказах.

**16. Что такое Raft?**  
Алгоритм консенсуса с ролями узлов и процедурой выбора лидера.

**17. Что такое Zab?**  
Протокол Apache ZooKeeper для согласования состояния и упорядочивания изменений.